---
title: "Lab 2: NumPy - obrazy rastrowe"
subtitle: "Biblioteki Python w analizie danych"
author: "Tomasz Rodak"
toc-title: "Spis treści"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_2.ipynb)

W tym arkuszu będziemy przetwarzać w NumPy obrazy rastrowe. Obrazy rastrowe to tablice pikseli, z których każdy ma przypisaną wartość koloru. W przypadku obrazów RGB każdy piksel ma trzy wartości: intensywność koloru czerwonego, zielonego i niebieskiego. Wartości te są zazwyczaj z zakresu 0–255, gdzie 0 oznacza brak danego koloru, a 255 — jego maksymalną intensywność. Zakres 0–255 to dokładnie jeden bajt, w NumPy reprezentowany jako `uint8`.

Pomostem między obrazem reprezentowanym jako tablica NumPy a obrazem wyświetlanym na ekranie będzie dla nas biblioteka [Pillow](https://pillow.readthedocs.io/en/stable/index.html) (`PIL` — *Python Imaging Library*).



In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


**Konwencja wyświetlania:** Do wyświetlania obrazów używamy `Image.fromarray()` — obiekt PIL wyświetla się automatycznie w Jupyterze jako ostatnie wyrażenie w komórce. Matplotlib (`plt.imshow()`, `plt.subplots()`) stosujemy tam, gdzie potrzebujemy porównań obok siebie, colormap (np. `cmap='gray'` dla tablic 2D) lub wykresów (histogramy).

## 1. Obraz jako tablica NumPy

### 1.1 Wczytanie obrazu

Pobierz dowolny kolorowy obraz RGB z internetu (np. [ten](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/HepaticaNobilisSLO_flower.JPG/1024px-HepaticaNobilisSLO_flower.JPG)) i wczytaj go za pomocą `Image.open()`. Możesz pobrać obraz ręcznie lub skorzystać z biblioteki `requests`:



In [ ]:
import requests

url = 'https://upload.wikimedia.org/wikipedia/commons/d/d3/HepaticaNobilisSLO_flower.JPG'
r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:143.0) Gecko/20100101 Firefox/143.0"}
    )
r.raise_for_status()
with open('kwiatek.jpg', 'wb') as f:
    f.write(r.content)


Wczytaj obraz i wyświetl go w notatniku.

1. Jakie są wymiary obrazu (`img.size`)?
2. Jaki jest tryb kolorów (`img.mode`)?
3. Jaki jest format pliku (`img.format`)?





### 1.2 Konwersja do tablicy NumPy

Przekształć obiekt `Image` do tablicy NumPy za pomocą `np.array()`.

1. Jaki jest kształt (`shape`) powstałej tablicy? Zinterpretuj każdy wymiar.
2. Jaki jest typ danych (`dtype`)?
3. Ile wymiarów (`ndim`) ma tablica?
4. Ile łącznie wartości (`size`) zawiera tablica?




### 1.3 Dostęp do kanałów

Każdy piksel obrazu RGB jest opisany trzema wartościami — kanałami R, G, B — zapisanymi w ostatnim wymiarze tablicy. Wyodrębnij poszczególne kanały i wyświetl je jako osobne obrazy w skali szarości obok siebie.

*Wskazówka:* Kanał czerwony to `img_array[:, :, 0]`, zielony — `img_array[:, :, 1]`, niebieski — `img_array[:, :, 2]`. Ponieważ pojedynczy kanał jest tablicą 2D, PIL nie wyświetli go poprawnie jako obraz RGB — tutaj wygodniej jest użyć `plt.subplots()` z `plt.imshow(..., cmap='gray')`, aby matplotlib potraktował tablicę 2D jako obraz szaroodcieniowy.




## 2. Konwersja do skali szarości

Konwersja obrazu RGB do skali szarości polega na zastąpieniu trzech wartości (R, G, B) każdego piksela jedną wartością jasności. Standardowy wzór, uwzględniający percepcję ludzkiego oka, to:

$$L = 0{,}2126 \cdot R + 0{,}7152 \cdot G + 0{,}0722 \cdot B$$

### 2.1 Implementacja wektoryzowana

Zaimplementuj konwersję RGB → skala szarości za pomocą jednej operacji NumPy, bez pętli. Wynik powinien być tablicą 2D typu `uint8`.

*Wskazówka:* Zdefiniuj wektor wag `w = np.array([0.2126, 0.7152, 0.0722])`. Broadcasting pozwoli pomnożyć tablicę o kształcie `(H, W, 3)` przez wektor o kształcie `(3,)` — jaka operacja da sumę ważoną po ostatniej osi?

Wyświetl wynik za pomocą `Image.fromarray()` (PIL automatycznie rozpoznaje tablicę 2D `uint8` jako obraz w skali szarości).




### 2.2 Porównanie z pętlą

Napisz drugą implementację konwersji — za pomocą podwójnej pętli `for` iterującej po pikselach. Zmierz czas wykonania obu wersji za pomocą `%%timeit` (lub `time.time()` jeśli obraz jest duży). Jaka jest różnica?




## 3. Operacje punktowe na obrazach

Operacje punktowe to przekształcenia, w których nowa wartość piksela zależy wyłącznie od jego starej wartości (a nie od sąsiadów).

### 3.1 Negatyw

Negatyw obrazu w skali szarości powstaje przez odwrócenie jasności: $L' = 255 - L$. Utwórz negatyw obrazu szaroodcieniowego z zadania 2. Wyświetl oryginał i negatyw obok siebie (tu wygodnie jest użyć `plt.subplots()` z `cmap='gray'`, albo złożyć oba obrazy obok siebie za pomocą `np.hstack()` i wyświetlić przez `Image.fromarray()`).



### 3.2 Progowanie (thresholding)

Progowanie zamienia obraz szaroodcieniowy na obraz binarny (czarno-biały):

$$L'(x,y) = \begin{cases} 255 & \text{jeśli } L(x,y) > t \\ 0 & \text{w przeciwnym razie} \end{cases}$$

gdzie $t$ to próg.

1. Zaimplementuj progowanie za pomocą porównania tablicowego (wynikiem porównania `gray > t` jest tablica boolowska — jak zamienić ją na `uint8`?).
2. Wyświetl wyniki dla kilku wartości progu: $t \in \{64, 128, 192\}$.

### 3.3 Automatyczny próg Otsu

Ręczny dobór progu wymaga eksperymentowania. Metoda Otsu wyznacza próg automatycznie na podstawie rozkładu jasności pikseli.

**Intuicja.** Próg $t$ dzieli piksele na dwie klasy: ciemne ($\leq t$) i jasne ($> t$). Dobry próg to taki, przy którym piksele w każdej klasie są do siebie *podobne* — mają małą wariancję jasności. Metoda Otsu szuka progu $t^*$ minimalizującego sumę ważonych wariancji obu klas:

$$\sigma_w^2(t) = \frac{n_0(t)}{N}\,\sigma_0^2(t) + \frac{n_1(t)}{N}\,\sigma_1^2(t)$$

gdzie $n_0(t)$ to liczba pikseli o jasności $\leq t$, $n_1(t) = N - n_0(t)$, a $\sigma_0^2(t)$, $\sigma_1^2(t)$ to wariancje jasności w każdej z klas.

**Algorytm.** Naiwne podejście wymagałoby dla każdego $t \in \{0, \ldots, 255\}$ przejrzenia wszystkich $N$ pikseli — koszt $O(255 \times N)$. Kluczowy trik polega na *kompresji danych do histogramu*: tablica 256 liczb zastępuje miliony pikseli, a wszystkie statystyki obliczamy z histogramu.

1. **Histogram.** Oblicz histogram jasności:
   ```python
   h = np.bincount(gray.ravel(), minlength=256)
   ```
   Teraz `h[i]` to liczba pikseli o jasności $i$, a cała informacja o obrazie mieści się w 256 liczbach.

2. **Wektor jasności.** Zdefiniuj `bins = np.arange(256)`.

3. **Sumy kumulacyjne.** Oblicz:
   ```python
   cum_n = np.cumsum(h)            # cum_n[t] = n_0(t) = sum h[0..t]
   cum_sum = np.cumsum(bins * h)   # cum_sum[t] = sum i*h[i] dla i=0..t
   ```
   Dzięki temu $n_0(t)$ i $\sum_{i=0}^{t} i \cdot h[i]$ są dostępne dla *wszystkich* progów jednocześnie — bez pętli po $t$.

4. **Statystyki klas.** Dla każdego progu $t$:
   $$n_0(t) = \texttt{cum\_n}[t], \qquad \mu_0(t) = \frac{\texttt{cum\_sum}[t]}{n_0(t)}$$
   Statystyki klasy 1 wynikają z różnicy z wartościami globalnymi (tzn. wartościami dla $t = 255$).

5. **Wariancje klas.** To najtrudniejszy krok. Potrzebujesz sumy kumulacyjnej $\sum_{i=0}^{t} h[i] \cdot i^2$, z której wariancja wynika ze wzoru $\sigma^2 = \overline{x^2} - \bar{x}^2$. Uważaj na progi, przy których $n_0(t) = 0$ lub $n_1(t) = 0$ — dzielenie przez zero można zabezpieczyć np. przez `np.where` lub dodanie małej wartości `eps`.

6. **Optymalny próg.** Wybierz $t^* = \arg\min_t \sigma_w^2(t)$ za pomocą `np.argmin`.

**Weryfikacja:**

- Wyświetl wykres $\sigma_w^2(t)$ w funkcji $t$ — powinien mieć wyraźne minimum.
- Zastosuj znaleziony próg $t^*$ do progowania obrazu i porównaj wynik z ręcznymi progami z zadania 3.2.

### 3.4 Rozciąganie histogramu

Rozciąganie histogramu (*contrast stretching*) poprawia kontrast obrazu, mapując zakres jasności $[\min L, \max L]$ na pełen zakres $[0, 255]$:

$$L' = \frac{L - \min L}{\max L - \min L} \cdot 255$$

Zaimplementuj tę operację na tablicy szaroodcieniowej. Wyświetl obraz przed i po rozciągnięciu za pomocą `Image.fromarray()`. Wyświetl histogramy jasności przed i po rozciągnięciu za pomocą `plt.hist()` (tu matplotlib jest naturalnym wyborem, bo rysujemy wykres, nie obraz).




## 4. Wymiana palety kolorów (broadcasting 3D)

Celem tego zadania jest zamiana kolorów obrazu RGB na najbliższe kolory z zadanej palety. To kluczowe ćwiczenie z broadcastingu na tablicach trójwymiarowych.

### 4.1 Definicja palety

Stwórz tablicę `paleta` o kształcie `(k, 3)`, gdzie `k` to liczba kolorów w palecie. Każdy wiersz zawiera wartości RGB jednego koloru. Możesz wybrać kolory dowolnie lub skorzystać z tablicy kolorów, np. [stąd](https://www.rapidtables.com/web/color/RGB_Color.html).

Przykładowa paleta 8-kolorowa:



In [ ]:
paleta = np.array([
    [0,   0,   0],    # czarny
    [255, 255, 255],  # biały
    [255, 0,   0],    # czerwony
    [0,   255, 0],    # zielony
    [0,   0,   255],  # niebieski
    [255, 255, 0],    # żółty
    [255, 0,   255],  # magenta
    [0,   255, 255],  # cyan
], dtype=np.uint8)



### 4.2 Reshaping

Zamień tablicę obrazu o kształcie `(H, W, 3)` na tablicę 2D o kształcie `(N, 3)`, gdzie `N = H * W`. Każdy wiersz to wartości RGB jednego piksela.




### 4.3 Obliczenie odległości (broadcasting)

Dla każdego piksela oblicz odległość euklidesową do każdego koloru z palety. Powinno to dać tablicę o kształcie `(N, k)`.

*Wskazówka:* Wykorzystaj broadcasting. Przeanalizuj transformację kształtów:
```
piksele:  (N, 3)    →  (N, 1, 3)
paleta:   (k, 3)    →  (1, k, 3)
różnica:               (N, k, 3)
kwadrat + suma po osi 2:  (N, k)
```

Użyj `np.newaxis` (lub `None`) do dodania wymiaru i `np.sum()` z parametrem `axis` do redukcji.




### 4.4 Mapowanie pikseli na paletę

Dla każdego piksela znajdź indeks najbliższego koloru w palecie (`np.argmin()` z odpowiednim `axis`). Następnie za pomocą indeksowania złożonego (*fancy indexing*) utwórz nową tablicę pikseli, w której każdy piksel ma wartości RGB najbliższego koloru z palety.

Przywróć kształt `(H, W, 3)` i wyświetl wynikowy obraz za pomocą `Image.fromarray()`.




### 4.5 Eksperymenty

1. Przetestuj palety o różnej liczbie kolorów: 2, 4, 8, 16, 32. Jak zmienia się jakość obrazu?
2. Zamiast ręcznej palety, wylosuj `k` kolorów z rozkładu jednostajnego na $[0, 255]^3$. Porównaj wynik z paletą dobraną ręcznie.




## 5. Generowanie wzorów

### 5.1 Szachownica

Napisz program, który utworzy obrazek szachownicy 8×8 pól na tle o wymiarach 500×500 pikseli.

Wymiary:
* pola — 50×50 pikseli, czarne i białe,
* tło — szarość o wartości 100,
* szachownica wycentrowana, czarne pole w lewym dolnym rogu.

*Wskazówka:* Utwórz tablicę wypełnioną wartością tła. Następnie za pomocą indeksowania z przypisaniem (`A[slice] = value`) wypełnij odpowiednie fragmenty tablicy. Alternatywnie: skorzystaj z `np.tile()` na wzorcu 2×2. Wyświetl wynik za pomocą `Image.fromarray()`.




### 5.2 Gradient

Wygeneruj obraz o wymiarach 256×256 pikseli przedstawiający gradient kolorów:
* w kierunku poziomym — od czarnego (0) do czerwonego (255, 0, 0),
* w kierunku pionowym — od czarnego (0) do niebieskiego (0, 0, 255).

W efekcie prawy dolny róg powinien być magentą (255, 0, 255).

*Wskazówka:* Utwórz tablicę `(256, 256, 3)` wypełnioną zerami. Kanał czerwony to funkcja kolumny, kanał niebieski — funkcja wiersza. Broadcasting z `np.arange()` lub `np.linspace()` pozwoli uniknąć pętli. Wyświetl wynik za pomocą `Image.fromarray()`.





## 6. Widoki i kopie

### 6.1 Modyfikacja in-place przez widok

1. Utwórz kopię tablicy obrazu (`img_copy = img_array.copy()`).
2. Za pomocą wycinków wyodrębnij prostokątny fragment obrazu (np. lewy górny kwadrant).
3. Sprawdź, czy wyodrębniony fragment jest widokiem (`np.shares_memory()`).
4. Ustaw wszystkie piksele fragmentu na kolor czerwony `[255, 0, 0]`. Czy oryginalna tablica `img_copy` też się zmieniła?




### 6.2 Indeksowanie złożone — kopia

1. Utwórz tablicę boolowską (maskę) wskazującą piksele, których kanał czerwony jest większy niż 200.
2. Wyodrębnij te piksele za pomocą maski. Czy wynik jest widokiem czy kopią? Sprawdź za pomocą `np.shares_memory()`.
3. Ustaw kolor wyodrębnionych pikseli na zielony. Czy oryginalna tablica się zmieniła?
4. Jak osiągnąć modyfikację oryginału za pomocą maski? (*Wskazówka:* `img_array[maska] = [0, 255, 0]`.)




## 7. Zadania dodatkowe

### 7.1 Wymiana kanałów

Utwórz wersję obrazu, w której kanały R i B są zamienione miejscami. Wyświetl wynik. Czy potrzebujesz kopii, czy wystarczy widok? Co się stanie, jeśli użyjesz `img_array[:, :, [2, 1, 0]]` a co jeśli napiszesz `img_array[:, :, ::-1]`? Sprawdź, które z tych podejść zwraca widok, a które kopię.




### 7.2 Mozaika z `np.tile()`

Utwórz mozaikę 3×3 z pomniejszonego obrazu. Aby pomniejszyć obraz, weź co `n`-ty piksel w obu kierunkach (np. `img_array[::4, ::4]`). Następnie użyj `np.tile()` do powielenia tablicy.

